In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_firm_structure_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :

        # -------DATE_FILTER
        date_filter = f"businessdate = {run_date!r}"

        # -------SCHEMA
        # schema = (
        #     "FirmStructureId string,"
        #     "SourceSystem string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "FirmStructureKey string,"
        #     "EmployeePartyKey string,"
        #     "EmployeeCategory string,"
        #     "EmployeeFirstName string,"
        #     "EmployeeLastName string,"
        #     "EmployeeFullName string,"
        #     "EmployeeEmail string,"
        #     "EmployeeUsername string,"
        #     "EmployeeUserTypeCode string,"
        #     "EmployeeUserType string,"
        #     "EmployeePositionCode string ,"
        #     "EmployeePosition string,"
        #     "DepartmentCode string,"
        #     "Department string,"
        #     "TeamCode string,"
        #     "Team string,"
        #     "SupervisorPartyKey string,"
        #     "SupervisorPartyName string,"
        #     "SupervisorUsername string,"
        #     "ManagerPartyKey string,"
        #     "ManagerPartyName string,"
        #     "ManagerUsername string,"
        #     "DirectorPartyKey string,"
        #     "DirectorPartyName string,"
        #     "DirectorUsername string,"
        #     "IsEmployeeActiveFlag boolean,"
        #     "CreatedDate date,"
        #     "LastUpdatedDate date,"
        #     "BusinessDate date,"
        # )

        cluster_columns = ["BusinessDate"]

        # -------CHOOSE PATH

        curated_table_path = f"{curated_zone}.FirmStructure"

        # -------CREATION OF DELTA TABLE
        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # --------APPLY FILTERS AND SELECT COLUMNS FROM THE TABLES WE NEED

        # ================== THESIS_SMUSER ==================

        smuser_extra_filters = None

        if smuser_extra_filters is not None:
            smuser_extra_filters.append(date_filter)
            smuser_final_filters = " AND ".join(smuser_extra_filters)
        else:
            smuser_final_filters = date_filter

        smuser_selected_fields = [
            "id",
            "name",
            "full_name",
            "email",
            "loginname",
            "role_code",
            "job_descr",
            "active",
            "BusinessDate",
        ]

        SMUSER = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_smuser",
            smuser_final_filters,
            smuser_selected_fields,
        )

        # --------SET THE KEYS FOR HASHKEY BASED ON DATA MAPPING
        Firm_StructureId_primary_key_fields = [
            SMUSER.id,
            lit("Employee"),
            lit("thesis"),
        ]

        Firm_StructureKey_primary_key_fields = [
            SMUSER.id,
            lit("Employee"),
        ]

        # -------CREATION OF HΑSKEY
        firm_structure_df = (
            SMUSER.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *Firm_StructureId_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey", concat_ws(";", *Firm_StructureKey_primary_key_fields)
        )

        # -------SELECTION OF FINAL COLUMNS
        result_df_thesis = firm_structure_df.select(
            firm_structure_df.FirmStructureId.alias("FirmStructureId"),
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit("None").alias("BatchId"),
            firm_structure_df.FirmStructureKey.cast("string").alias("FirmStructureKey"),
            SMUSER.id.cast("string").alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            SMUSER.name.alias("EmployeeFirstName"),
            lit("None").alias("EmployeeLastName"),
            SMUSER.full_name.alias("EmployeeFullName"),
            SMUSER.email.alias("EmployeeEmail"),
            SMUSER.loginname.alias("EmployeeUsername"),
            lit("None").alias("EmployeeUserTypeCode"),
            lit("None").alias("EmployeeUserType"),
            SMUSER.role_code.alias("EmployeePositionCode"),
            SMUSER.job_descr.alias("EmployeePosition"),
            lit("None").alias("DepartmentCode"),
            lit("None").alias("Department"),
            lit("None").alias("TeamCode"),
            lit("None").cast("string").alias("Team"),
            lit("None").cast("string").alias("SupervisorPartyKey"),
            lit("None").alias("SupervisorPartyName"),
            lit("None").alias("SupervisorUsername"),
            lit("None").cast("string").alias("ManagerPartyKey"),
            lit("None").alias("ManagerPartyName"),
            lit("None").alias("ManagerUsername"),
            lit("None").cast("string").alias("DirectorPartyKey"),
            lit("None").alias("DirectorPartyName"),
            lit("None").alias("DirectorUsername"),
            SMUSER.active.alias("IsEmployeeActiveFlag"),
            lit("None").cast("date").alias("CreatedDate"),
            lit("None").cast("date").alias("LastUpdatedDate"),
            SMUSER.BusinessDate.alias("BusinessDate"),
        )

        # --------------APPEND DATA TO CURATED
        result_df_thesis.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_firm_structure_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)